# Backend Fallback Cases Demo

**v0.4.8 — canonical unsupported/native-fallback example**

This notebook demonstrates the two categories of workflows that **cannot** use RamanSPy
and must stay on the native backend:

| Section | Reason |
|---------|--------|
| **A: Raman + unsupported step** | `BaselineSubtract(method="gaussian")` is not yet translated |
| **B: PL workflow** | PL modality is native-only by design |

In `auto` mode, these fall back gracefully. In forced `ramanspy` mode, they raise
`NotImplementedError` with a clear message.

Understanding these cases helps distinguish:
- supported Raman execution (see `Ramanfit/Raman_backend_demo.ipynb`),
- automatic fallback (shown here), and
- forced-backend failure (shown here).

In [1]:
import numpy as np
import sys, os

from ramanpl import RamanFit, PLfit
from ramanpl.dataImporter import DataImporter
from ramanpl.preprocessing import Pipeline, CropByRange, SmoothSavGol, BaselineSubtract

---
## Section A: Raman + Gaussian baseline (unsupported preprocessing step)

`BaselineSubtract(method="gaussian")` is not yet translatable to a RamanSPy operation.
This makes the pipeline **not fully supported**, so:
- `auto` falls back to native (with a reason recorded)
- forced `ramanspy` raises `NotImplementedError`

This is **expected behaviour**, not a bug.

In [2]:
# Load a Raman spectrum (change path if running from a different directory)
spectra, wavenumber = DataImporter.data_import(
    filename=os.path.join("..", "Ramanfit", "Raman Sample MoS2 2D Flakes.wdf"),
    x_range=(250, 750),
)
print(f"Loaded: {wavenumber.size} points, {wavenumber[0]:.0f}–{wavenumber[-1]:.0f} cm⁻¹")

Loaded: 270 points, 749–252 cm⁻¹


### A1. `auto` falls back to native (Gaussian baseline is not translatable)

In [3]:
pipe_gaussian_auto = Pipeline(
    steps=[
        CropByRange((300, 700)),
        BaselineSubtract({"method": "gaussian", "gaussian_sigma": 50}),
    ],
    backend="auto",
)

fit_gaussian_auto = RamanFit.RamanFit(
    spectra=spectra,
    wavenumber=wavenumber,
    materials=['MoS2'],
    substrate='Si',
    normalize=False,
    preprocessing=pipe_gaussian_auto,
)
fit_gaussian_auto.fit_spectrum()

outcome = fit_gaussian_auto._backend_outcome
print("requested :", outcome["requested_backend"])
print("resolved  :", outcome["resolved_backend"])
print("fallback  :", outcome["fallback_used"])
print("reason    :", outcome.get("fallback_reason"))

requested : auto
resolved  : native
fallback  : True
reason    : Automatic fallback to native: RamanSPy translation is not yet implemented for: ["BaselineSubtract(method='gaussian') remains native-only. RamanSPy exposes Gaussian denoising, not the same baseline estimator."]


### A2. Forced `ramanspy` raises `NotImplementedError`

When you force `backend="ramanspy"` with a pipeline that contains a non-translatable step,
the error message names the unsupported step explicitly.

In [4]:
pipe_gaussian_forced = Pipeline(
    steps=[
        CropByRange((300, 700)),
        BaselineSubtract({"method": "gaussian", "gaussian_sigma": 50}),
    ],
    backend="ramanspy",
)

try:
    RamanFit.RamanFit(
        spectra=spectra,
        wavenumber=wavenumber,
        materials=['MoS2'],
        substrate='Si',
        normalize=False,
        preprocessing=pipe_gaussian_forced,
    )
except NotImplementedError as e:
    print("NotImplementedError raised as expected:")
    print(e)

NotImplementedError raised as expected:
RamanSPy translation is currently implemented for CropByRange, SmoothSavGol, and BaselineSubtract (poly / asls / airpls / arpls). Unsupported steps: ["BaselineSubtract(method='gaussian') remains native-only. RamanSPy exposes Gaussian denoising, not the same baseline estimator."]


---
## Section B: PL workflow (native-only by modality)

PL spectra are not expressed in cm⁻¹ Raman shift, so RamanSPy preprocessing does not
apply to them. The backend always resolves to `native` for PL workflows, regardless of
the `preprocessing_backend` setting.

This is **by design**: PL fitting remains on the native path in all v0.4.x builds.

In [5]:
# Load a PL spectrum
intensity, energy = DataImporter.data_import(
    filename=os.path.join("..", "PLfit", "PL Sample MoS2 2D Flakes.wdf"),
    x_range=(1.74, 1.88),
    axis="energy",
)
print(f"Loaded PL spectrum: {energy.size} points, {energy[0]:.3f}–{energy[-1]:.3f} eV")

Loaded PL spectrum: 991 points, 1.744–1.880 eV


### B1. `auto` stays native for PL

In [6]:
pl_pipe_auto = Pipeline(
    steps=[SmoothSavGol(window_length=11, polyorder=3)],
    backend="auto",
)

pl_fit_auto = PLfit.PLfit(
    intensity, energy,
    background_remove=False,
    smoothing=False,
    normalize=False,
    preprocessing=pl_pipe_auto,
)
pl_fit_auto.update_bounds(
    exciton=([1.82, 0, 0], [1.88, 0.05, 0.1]),
    trion=  ([1.80, 0, 0], [1.86, 0.05, 0.05]),
)
pl_fit_auto.fit_spectrum()

outcome_pl = pl_fit_auto._backend_outcome
print("requested :", outcome_pl["requested_backend"])
print("resolved  :", outcome_pl["resolved_backend"])
print("fallback  :", outcome_pl["fallback_used"])
print("reason    :", outcome_pl.get("fallback_reason"))

requested : auto
resolved  : native
fallback  : True
reason    : Automatic fallback to native: input modality or axis is not supported by RamanSPy


### B2. Forced `ramanspy` raises for PL input

In [7]:
pl_pipe_forced = Pipeline(
    steps=[SmoothSavGol(window_length=11, polyorder=3)],
    backend="ramanspy",
)

try:
    PLfit.PLfit(
        intensity, energy,
        background_remove=False,
        smoothing=False,
        normalize=False,
        preprocessing=pl_pipe_forced,
    )
except (NotImplementedError, ValueError) as e:
    print(f"{type(e).__name__} raised as expected:")
    print(e)

NotImplementedError raised as expected:
RamanSPy backend currently supports Raman workflows only.


---
## Summary

| Scenario | Backend requested | Backend resolved | Notes |
|----------|-------------------|-----------------|-------|
| Raman + Gaussian baseline | `auto` | `native` | fallback, reason recorded |
| Raman + Gaussian baseline | `ramanspy` | — | `NotImplementedError` |
| PL + supported steps | `auto` | `native` | PL is native-only by design |
| PL + supported steps | `ramanspy` | — | error raised |

For workflows where RamanSPy **is** used, see:
- [`../Ramanfit/Raman_backend_demo.ipynb`](../Ramanfit/Raman_backend_demo.ipynb) — single-spectrum
- [`../Mapping/Raman_mapping_backend_demo.ipynb`](../Mapping/Raman_mapping_backend_demo.ipynb) — mapping